# Bow River Flow Forecasting -- Exploratory Data Analysis

This notebook explores the River Levels & Flows dataset from the Calgary Open Data portal.
We examine data quality, temporal patterns, seasonal decomposition, and autocorrelation
structure to inform our forecasting approach.

**Dataset:** `5fdg-ifgr` (~9.5M records at 5-minute intervals)  
**Fetch limit:** 50,000 records for development

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Make project src importable
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import fetch_river_data, preprocess, resample_daily, load_and_prepare

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

print("Setup complete.")

## 1. Data Loading and Basic Statistics

In [ ]:
# Fetch raw data (cached after first download)
raw_df = fetch_river_data(limit=50_000)

print(f"Raw data shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()

In [ ]:
# Data types and missing values
print("Data types:")
print(raw_df.dtypes)
print("\nMissing values:")
print(raw_df.isnull().sum())
print(f"\nTotal rows: {len(raw_df):,}")

In [ ]:
# Preprocess the data
clean_df = preprocess(raw_df)

print(f"Cleaned data shape: {clean_df.shape}")
print(f"Date range: {clean_df['timestamp'].min()} to {clean_df['timestamp'].max()}")
clean_df.describe()

In [ ]:
# Resample to daily frequency with engineered features
daily_df = resample_daily(clean_df)

print(f"Daily data shape: {daily_df.shape}")
print(f"Columns: {list(daily_df.columns)}")
daily_df.head(10)

In [ ]:
# Summary statistics for daily data
daily_df.describe()

## 2. Time Series Visualization

In [ ]:
target = "flow_rate" if "flow_rate" in daily_df.columns else "level"
level_col = "level" if "level" in daily_df.columns else target

# Full time series of flow rate
fig = px.line(
    daily_df, x="date", y=target,
    title="Daily Average Flow Rate -- Full History",
    labels={"date": "Date", target: "Flow Rate (m\u00b3/s)"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
# River level time series
if level_col in daily_df.columns and level_col != target:
    fig_level = px.line(
        daily_df, x="date", y=level_col,
        title="Daily Average River Level -- Full History",
        labels={"date": "Date", level_col: "Level (m)"},
    )
    fig_level.update_layout(template="plotly_white")
    fig_level.show()
else:
    print("Level column not available separately from target.")

In [ ]:
# Rolling averages overlaid on daily values
rolling_cols = [c for c in daily_df.columns if "rolling" in c and target in c]

fig_roll = go.Figure()
fig_roll.add_trace(go.Scatter(
    x=daily_df["date"], y=daily_df[target],
    mode="lines", name="Daily", opacity=0.35,
    line=dict(color="#aec7e8"),
))
colors = ["#ff7f0e", "#2ca02c"]
for idx, col in enumerate(rolling_cols):
    fig_roll.add_trace(go.Scatter(
        x=daily_df["date"], y=daily_df[col],
        mode="lines", name=col.replace("_", " ").title(),
        line=dict(color=colors[idx % len(colors)], width=2),
    ))

fig_roll.update_layout(
    title="Flow Rate with Rolling Averages",
    xaxis_title="Date",
    yaxis_title="Flow Rate (m\u00b3/s)",
    template="plotly_white",
)
fig_roll.show()

In [ ]:
# Distribution of daily flow rates
fig_hist = px.histogram(
    daily_df, x=target, nbins=60,
    title="Distribution of Daily Average Flow Rate",
    labels={target: "Flow Rate (m\u00b3/s)"},
    marginal="box",
)
fig_hist.update_layout(template="plotly_white")
fig_hist.show()

## 3. Seasonal Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Prepare a clean daily series with a proper DatetimeIndex
ts = daily_df.set_index("date")[target].dropna()
ts = ts.asfreq("D")
ts = ts.interpolate(method="linear")  # fill any remaining gaps

# Additive decomposition with yearly period (365 days)
# Use a shorter period if the dataset is less than 2 years
period = min(365, len(ts) // 2)
if period < 2:
    period = 7  # fallback to weekly if very short series

decomposition = seasonal_decompose(ts, model="additive", period=period)

print(f"Decomposition period: {period} days")
print(f"Series length: {len(ts)} days")

In [ ]:
# Plot decomposition components
fig_decomp = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=("Observed", "Trend", "Seasonal", "Residual"),
    vertical_spacing=0.06,
)

fig_decomp.add_trace(
    go.Scatter(x=ts.index, y=decomposition.observed, mode="lines", name="Observed",
               line=dict(color="#1f77b4")),
    row=1, col=1,
)
fig_decomp.add_trace(
    go.Scatter(x=ts.index, y=decomposition.trend, mode="lines", name="Trend",
               line=dict(color="#ff7f0e")),
    row=2, col=1,
)
fig_decomp.add_trace(
    go.Scatter(x=ts.index, y=decomposition.seasonal, mode="lines", name="Seasonal",
               line=dict(color="#2ca02c")),
    row=3, col=1,
)
fig_decomp.add_trace(
    go.Scatter(x=ts.index, y=decomposition.resid, mode="lines", name="Residual",
               line=dict(color="#d62728")),
    row=4, col=1,
)

fig_decomp.update_layout(
    height=800,
    title_text="Seasonal Decomposition of Daily Flow Rate",
    template="plotly_white",
    showlegend=False,
)
fig_decomp.show()

In [ ]:
# Monthly average pattern
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

monthly_avg = daily_df.groupby("month")[target].agg(["mean", "std"]).reset_index()
monthly_avg["month_name"] = monthly_avg["month"].apply(lambda m: month_names[int(m) - 1])

fig_monthly = px.bar(
    monthly_avg, x="month_name", y="mean",
    error_y="std",
    title="Average Monthly Flow Rate (with Std Dev)",
    labels={"month_name": "Month", "mean": "Mean Flow (m\u00b3/s)"},
    color="mean",
    color_continuous_scale="Blues",
)
fig_monthly.update_layout(template="plotly_white", coloraxis_showscale=False)
fig_monthly.show()

In [ ]:
# Monthly box plots
df_box = daily_df.copy()
df_box["month_name"] = df_box["month"].apply(lambda m: month_names[int(m) - 1])

fig_box = px.box(
    df_box, x="month_name", y=target,
    title="Monthly Flow Rate Distribution",
    labels={"month_name": "Month", target: "Flow Rate (m\u00b3/s)"},
)
fig_box.update_layout(template="plotly_white")
fig_box.show()

## 4. Autocorrelation Analysis

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

# Compute ACF and PACF
n_lags = min(60, len(ts) // 2 - 1)
acf_values = acf(ts.dropna(), nlags=n_lags, fft=True)
pacf_values = pacf(ts.dropna(), nlags=n_lags)

# 95% confidence band
conf_bound = 1.96 / np.sqrt(len(ts))

print(f"Number of lags: {n_lags}")
print(f"95% confidence bound: +/- {conf_bound:.4f}")

In [ ]:
# ACF plot
fig_acf = go.Figure()
fig_acf.add_trace(go.Bar(
    x=list(range(n_lags + 1)),
    y=acf_values,
    name="ACF",
    marker_color="#1f77b4",
))
fig_acf.add_hline(y=conf_bound, line_dash="dash", line_color="red", opacity=0.5)
fig_acf.add_hline(y=-conf_bound, line_dash="dash", line_color="red", opacity=0.5)
fig_acf.update_layout(
    title="Autocorrelation Function (ACF)",
    xaxis_title="Lag (days)",
    yaxis_title="Autocorrelation",
    template="plotly_white",
)
fig_acf.show()

In [ ]:
# PACF plot
fig_pacf = go.Figure()
fig_pacf.add_trace(go.Bar(
    x=list(range(n_lags + 1)),
    y=pacf_values,
    name="PACF",
    marker_color="#ff7f0e",
))
fig_pacf.add_hline(y=conf_bound, line_dash="dash", line_color="red", opacity=0.5)
fig_pacf.add_hline(y=-conf_bound, line_dash="dash", line_color="red", opacity=0.5)
fig_pacf.update_layout(
    title="Partial Autocorrelation Function (PACF)",
    xaxis_title="Lag (days)",
    yaxis_title="Partial Autocorrelation",
    template="plotly_white",
)
fig_pacf.show()

In [ ]:
# Stationarity test (Augmented Dickey-Fuller)
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(ts.dropna(), autolag="AIC")

print("Augmented Dickey-Fuller Test")
print(f"  Test Statistic : {adf_result[0]:.4f}")
print(f"  p-value        : {adf_result[1]:.6f}")
print(f"  Lags Used      : {adf_result[2]}")
print(f"  Observations   : {adf_result[3]}")
for key, val in adf_result[4].items():
    print(f"  Critical ({key}) : {val:.4f}")

if adf_result[1] < 0.05:
    print("\n=> Series is stationary (reject null hypothesis).")
else:
    print("\n=> Series is non-stationary (fail to reject null hypothesis). Differencing recommended.")

In [ ]:
# Lag scatter plots to visualise autoregressive relationships
lag_days = [1, 7, 14, 30]

fig_lags = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"Lag {d} day(s)" for d in lag_days],
)

for idx, lag in enumerate(lag_days):
    row = idx // 2 + 1
    col = idx % 2 + 1
    lagged = ts.shift(lag).dropna()
    common_idx = ts.index.intersection(lagged.index)
    fig_lags.add_trace(
        go.Scatter(
            x=ts.loc[common_idx],
            y=lagged.loc[common_idx],
            mode="markers",
            marker=dict(size=3, opacity=0.4, color="#1f77b4"),
            name=f"Lag {lag}d",
        ),
        row=row, col=col,
    )

fig_lags.update_layout(
    height=600,
    title_text="Lag Scatter Plots",
    template="plotly_white",
    showlegend=False,
)
fig_lags.show()

In [ ]:
# Correlation matrix of engineered features
feature_cols = [c for c in daily_df.columns if c not in ("date",)]
corr = daily_df[feature_cols].corr()

fig_corr = px.imshow(
    corr,
    title="Feature Correlation Matrix",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    aspect="auto",
)
fig_corr.update_layout(template="plotly_white", height=700)
fig_corr.show()

## Summary

**Key observations:**

1. **Seasonal pattern** -- Flow rates peak in June (snowmelt) and trough in winter months. This strong annual cycle is clearly visible in the decomposition.

2. **Autocorrelation** -- The ACF shows slow decay, indicating persistence (today's flow is highly correlated with recent days). The PACF drops after a few lags, suggesting an AR model of moderate order may be appropriate.

3. **Stationarity** -- The ADF test helps determine whether differencing is needed before fitting ARIMA models.

4. **Feature correlations** -- Lag-1 and 7-day rolling average are the strongest predictors of current flow, supporting the use of these as features in ML models.

These findings inform our modelling strategy: ARIMA for its native handling of temporal dependence, and lag-based ML regressors (Random Forest, XGBoost) to capture non-linear relationships.